# Chapter 7 — Evaluation v2: SFT v1, DPO v1, SFT v2, DPO v2

Four questions, each answered with paired tests on conversations **no model was trained on**:

1. **Did DPO v2 improve on SFT v2?**
2. **Is the v2 pipeline better than the old final model** (DPO v1 = `dpo_v7_rpo`)?
3. **Is it safe to show?** Private names, placeholders, loops, fallbacks.
4. **How far is it from a real person?** How often a judge still prefers your real reply.

### Three kinds of evidence
| Evidence | Measures | Why trust it |
|---|---|---|
| **Likelihood** of your real reply, and **preference accuracy** (real reply vs a wrong one) | How well each model knows what you actually say | No judge, no opinion |
| **Automatic checks** on generated replies | Length vs your real length, rambling, placeholders, names, generic-assistant phrasing | Transparent counts |
| **Local LLM judge** (Qwen3-14B, runs on this GPU) | Which reply makes more sense here, and which is closer to your real reply | **Validated first**: it must pick your real reply over a wrong one often enough, or its verdicts are not used |

**Nothing leaves Colab**: the judge is an open model downloaded to this runtime, so no conversation is sent to any API.

### How to run
- Runtime → **T4 GPU** → Run all. About **60–80 minutes**: four models generate, then the judge. Every stage is saved as it goes, so after a disconnect: Run all again and it continues.
- Needs on Drive: `ahmed_twin/model_merged_nilechat_v2`, the DPO v2 adapter and pairs, `private_names.txt`. The v1 models are **optional**: if `model_merged_nilechat` or `DPO/runs/dpo_v7_rpo/adapter` is not visible from this account, those comparisons are skipped.
- `eval_summary.json` holds numbers only (shareable). `eval_private.jsonl` holds real messages (never share). **Before committing: Edit → Clear all outputs.**

### Why this design (three rounds)

| Round | Design | What was wrong with it |
|---|---|---|
| 1 | The v1 notebook with new paths: SFT v2 vs DPO v2, greedy decoding, Cerebras judge on anonymized chats | Real chats sent to an external API; only 2 models, so "is v2 better than the old final model?" stays unanswered; greedy decoding is not what the twin really uses; the judge is never checked; `[NAME]` in v2 data is unfair to v1, which never saw it |
| 2 | Four systems; the deployed decoding (temperature 0.4 + reply filter); a local judge that *writes* A/B/TIE; two judge questions (makes sense / like the real reply); Holm correction | A 14B judge writing text is slow and its answers need parsing; still no proof the judge can tell good from bad; v1's training data never checked against the test conversations; 6 pairwise comparisons is too many for a T4 |
| **3 (this)** | The judge **reads its own probabilities** for A and B (one batched forward pass, no parsing); a **calibration test** gates the judge (real reply vs a wrong reply); **v1 leakage check** (name-insensitive); fair subset for v1; only the comparisons that answer the four questions; same seed for every system; optional blind human check that measures judge-human agreement | Remaining limits are listed at the end |

## A. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q peft bitsandbytes
!pip uninstall -y -q torchao  # Colab's old torchao makes peft refuse to load; nothing here uses it

### A1. Settings: the only cell you should need to edit

`SYSTEMS` maps each model to its base folder and its adapter (or `None`). Systems in `OPTIONAL_SYSTEMS` are skipped if their folders are missing.

In [ ]:
import contextlib
import gc
import hashlib
import json
import math
import os
import random
import re
import statistics
from collections import Counter

import peft
import transformers
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

EVAL_RUN_NAME = "ch7_v2"
DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/ahmed_twin"

SYSTEMS = {   # name -> (merged base model, adapter or None)
    "sft_v1": (f"{PROJECT_DIR}/model_merged_nilechat", None),
    "dpo_v1": (f"{PROJECT_DIR}/model_merged_nilechat", f"{DRIVE_ROOT}/DPO/runs/dpo_v7_rpo/adapter"),
    "sft_v2": (f"{PROJECT_DIR}/model_merged_nilechat_v2", None),
    "dpo_v2": (f"{PROJECT_DIR}/model_merged_nilechat_v2", f"{PROJECT_DIR}/dpo_v2/runs/dpo_v2_beta02_sft05/adapter"),
}
OPTIONAL_SYSTEMS = {"sft_v1", "dpo_v1"}
PAIRS_DIR = f"{PROJECT_DIR}/dpo_v2/pairs"           # test.jsonl: conversations unseen by every v2 stage
V1_TRAIN_PATH = f"{PROJECT_DIR}/train.jsonl"        # v1's training data: any test conversation found in it is removed
PRIVATE_NAMES_PATH = f"{PROJECT_DIR}/private_names.txt"
EVAL_DIR = f"{PROJECT_DIR}/eval_v2/{EVAL_RUN_NAME}"

SEED = 0
N_EVAL = 200                 # conversations the models answer
MAX_PER_CONVERSATION = 4     # at most this many exchanges from one conversation, so no single chat dominates
N_PREFERENCE_PAIRS = 600     # real reply vs wrong reply, scored by likelihood
DECODING = dict(do_sample=True, temperature=0.4, max_new_tokens=60)   # what the twin really uses
FILTER_TRIES = 3

RUN_JUDGE = True
JUDGE_MODEL = "unsloth/Qwen3-14B-unsloth-bnb-4bit"   # open, already 4-bit (~11 GB download); runs on this GPU
JUDGE_LOAD_4BIT = False      # True only for a judge that is NOT already quantized
JUDGE_CONTEXT_TURNS = 6
JUDGE_BATCH = 4
JUDGE_PROMPT_VERSION = 1     # bump after editing the judge prompts (old verdicts are then not reused)
MIN_JUDGE_CALIBRATION = 0.65 # the judge must pick the real reply over a wrong one this often to be trusted

SHOW_EXAMPLES = False        # True prints real conversations: private, clear outputs afterwards
HUMAN_CHECK_SIZE = 30
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| peft", peft.__version__)

### A2. Available systems and the run folder

A required system that is missing stops the notebook; an optional one is skipped with a note. The run's settings are saved the first time; a later run with the same `EVAL_RUN_NAME` and different settings is refused, so results from two settings never mix. The judge has its own fingerprint: changing the judge never forces the models to generate again.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


ACTIVE = {}
for name, (base, adapter) in SYSTEMS.items():
    present = os.path.exists(os.path.join(base, "config.json")) and (
        adapter is None or os.path.exists(os.path.join(adapter, "adapter_config.json")))
    if present:
        ACTIVE[name] = (base, adapter)
    elif name in OPTIONAL_SYSTEMS:
        print(f"Skipping {name}: {base if not os.path.exists(base) else adapter} is not visible from this account.")
    else:
        raise FileNotFoundError(f"Required system {name} is missing: {base} / {adapter}")
assert "sft_v2" in ACTIVE and "dpo_v2" in ACTIVE, "SFT v2 and DPO v2 are required."
print("Systems:", ", ".join(ACTIVE))

MANIFEST_PATH = f"{PAIRS_DIR}/manifest.json"
GEN_CONFIG = json.loads(json.dumps({
    "systems": ACTIVE, "pairs_manifest_sha256": sha256_of(MANIFEST_PATH), "seed": SEED, "n_eval": N_EVAL,
    "max_per_conversation": MAX_PER_CONVERSATION, "n_preference_pairs": N_PREFERENCE_PAIRS,
    "decoding": DECODING, "filter_tries": FILTER_TRIES, "v1_train_checked": os.path.exists(V1_TRAIN_PATH),
}))
JUDGE_TAG = f"{re.sub(r'[^A-Za-z0-9]+', '-', JUDGE_MODEL)}_p{JUDGE_PROMPT_VERSION}_c{JUDGE_CONTEXT_TURNS}"
CONFIG_PATH = f"{EVAL_DIR}/eval_config.json"
SAMPLE_PATH = f"{EVAL_DIR}/eval_sample_private.json"
SUMMARY_PATH = f"{EVAL_DIR}/eval_summary.json"
PRIVATE_PATH = f"{EVAL_DIR}/eval_private.jsonl"
os.makedirs(EVAL_DIR, exist_ok=True)
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, encoding="utf-8") as f:
        if json.load(f) != GEN_CONFIG:
            raise ValueError(f"{EVAL_DIR} was started with different settings or systems. Use a new EVAL_RUN_NAME.")
    print("Resuming:", EVAL_DIR)
else:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(GEN_CONFIG, f, indent=2)
    print("New evaluation run:", EVAL_DIR)

### A3. Private names (same rules as `tools/private_names_lib.py`) and the reply filter (`tools/reply_filter.py`, unchanged)

In [ ]:
ARABIC = "؀-ۿ"
NAME_PREFIXES = "وبل"


def load_private_names(path):
    with open(path, encoding="utf-8") as f:
        names = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]
    return sorted(set(names), key=len, reverse=True)


def compile_name_patterns(names):
    arabic = [n for n in names if re.search(f"[{ARABIC}]", n)]
    latin = [n for n in names if n not in arabic]
    ar = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, arabic))})(?![{ARABIC}])")
          if arabic else None)
    la = re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, latin))})(?![A-Za-z])", re.IGNORECASE) if latin else None
    return ar, la


def find_private_names(text, patterns):
    ar, la = patterns
    return ([m.group("name") for m in ar.finditer(text)] if ar else []) + ([m.group(0) for m in la.finditer(text)] if la else [])


NAME_PATTERNS = compile_name_patterns(load_private_names(PRIVATE_NAMES_PATH)) if os.path.exists(PRIVATE_NAMES_PATH) else None
find_names = (lambda text: find_private_names(text, NAME_PATTERNS)) if NAME_PATTERNS else None
print("Name check:", "on" if NAME_PATTERNS else "OFF (private_names.txt not found)")

In [ ]:
"""Output filter for the twin's generated replies (standard library only).

The SFT v2 model learned from redacted data, so it sometimes writes the placeholders themselves ("[NAME]",
"[PHONE]"); a sampled reply can also loop or contain a long number. This is the last line of defence between
the model and a reader: clean the reply, check it, and if it is not safe, generate again; after a few tries,
answer with a natural deflection instead.

The test notebooks embed this file's code unchanged (tests/test_reply_filter.py checks it), so the evaluation,
any demo and these tests all use the same rules.
"""

import random
import re
from typing import Callable, Optional, Sequence

# A placeholder from the redaction, with the vocative "يا" before it ("يا [NAME]" -> "").
PLACEHOLDER = re.compile(r"\s*(?:يا\s*)?\[(?:NAME|PHONE|EMAIL|URL|NUMBER|CREDENTIAL|REDACTED)\]")
LONG_NUMBER = re.compile(r"[0-9٠-٩]{6,}")          # phone-like, code-like or account-like
REPEATED_CHARACTER = re.compile(r"(.)\1{7,}")       # the same character 8+ times in a row
REPEATED_WORD = re.compile(r"(\S+)(?:\s+\1){3,}")   # the same word 4+ times in a row
FALLBACKS = ("مش هقولك 😂", "سيبك من الموضوع ده", "بعدين نتكلم")


def clean_reply(text: str) -> str:
    """Removes placeholders, empty lines and a line repeated right after itself."""
    lines: list[str] = []
    for line in text.split("\n"):
        line = PLACEHOLDER.sub("", line).strip()
        if line and (not lines or line != lines[-1]):
            lines.append(line)
    return "\n".join(lines)


def problems(reply: str, find_names: Optional[Callable[[str], Sequence[str]]] = None) -> list[str]:
    """Reasons a cleaned reply must not be shown; an empty list means it is safe."""
    found = []
    if not reply.strip():
        found.append("empty")
    if find_names and find_names(reply):
        found.append("private_name")
    if LONG_NUMBER.search(reply):
        found.append("long_number")
    if REPEATED_CHARACTER.search(reply):
        found.append("repeated_characters")
    lines = [line for line in reply.split("\n") if line.strip()]
    if REPEATED_WORD.search(reply) or any(lines.count(line) >= 3 for line in set(lines)):
        found.append("repeated_text")
    return found


def safe_reply(generate: Callable[[], str], find_names: Optional[Callable[[str], Sequence[str]]] = None,
               tries: int = 3, rng: Optional[random.Random] = None,
               fallbacks: Sequence[str] = FALLBACKS) -> tuple[str, dict]:
    """Calls `generate` up to `tries` times and returns the first cleaned reply with no problems, or a fallback.

    The second value says what happened: attempts used, whether the fallback was used, and the problems of the
    rejected attempts (reasons only, never the text)."""
    rejected = []
    for attempt in range(1, tries + 1):
        reply = clean_reply(generate())
        found = problems(reply, find_names)
        if not found:
            return reply, {"attempts": attempt, "fallback": False, "rejected": rejected}
        rejected.append(found)
    return (rng or random).choice(list(fallbacks)), {"attempts": tries, "fallback": True, "rejected": rejected}

## B. The test conversations

- **Source:** `dpo_v2/pairs/test.jsonl`: validation conversations that SFT v2 and DPO v2 never trained on, and not the dev conversations that picked DPO v2's checkpoint. Each row has the context, **your real reply** and a **wrong reply** (yours, from another conversation).
- **v1 leakage check:** v1 was trained on an older, unredacted build. Any test conversation that also appears in v1's `train.jsonl` is removed. The match ignores names, placeholders and digits, so redaction cannot hide a copy.
- **Fair subset:** v1 never saw `[NAME]`-style placeholders, so comparisons involving v1 use only conversations without them.
- **Spread:** at most `MAX_PER_CONVERSATION` exchanges per conversation. The sample is frozen to disk, so a resumed run uses exactly the same conversations.

In [ ]:
PLACEHOLDER_TOKEN = re.compile(r"\[[A-Z]+\]")


def leak_key(texts):
    # The same exchange before and after redaction gives the same key.
    joined = " ".join(texts)
    if NAME_PATTERNS:
        ar, la = NAME_PATTERNS
        joined = ar.sub(lambda m: m.group("pre") or "", joined) if ar else joined
        joined = la.sub("", joined) if la else joined
    joined = PLACEHOLDER_TOKEN.sub("", joined)
    return re.sub(r"[\s0-9٠-٩]+", "", joined)


def has_placeholder(record):
    return any(PLACEHOLDER_TOKEN.search(t) for t in [m["content"] for m in record["messages"]] + [record["real"]])


if os.path.exists(SAMPLE_PATH):
    with open(SAMPLE_PATH, encoding="utf-8") as f:
        SAMPLE = json.load(f)
    print("Loaded the frozen sample:", SAMPLE_PATH)
else:
    unique = {}
    for pair in load_jsonl(f"{PAIRS_DIR}/test.jsonl"):
        record = {"id": pair["id"], "group": pair["group"], "messages": pair["prompt_messages"],
                  "real": pair["chosen"], "negative": pair["rejected"]}
        unique.setdefault(json.dumps(record["messages"], ensure_ascii=False), record)
    records = list(unique.values())

    leak = {"checked": False, "removed": 0}
    if os.path.exists(V1_TRAIN_PATH):
        v1_keys = {leak_key([m["content"] for m in r["messages"] if m["role"] != "system"]) for r in load_jsonl(V1_TRAIN_PATH)}
        kept = [r for r in records if leak_key([m["content"] for m in r["messages"][1:]] + [r["real"]]) not in v1_keys]
        leak = {"checked": True, "removed": len(records) - len(kept)}
        records = kept
    else:
        print("WARNING: v1's train.jsonl not found: v1 comparisons may be optimistic for v1.")

    rng = random.Random(SEED)
    by_group = {}
    for record in sorted(records, key=lambda r: r["id"]):
        by_group.setdefault(record["group"], []).append(record)
    capped = []
    for group in sorted(by_group):
        rows = by_group[group]
        rng.shuffle(rows)
        capped.extend(rows[:MAX_PER_CONVERSATION])
    generation = sorted(rng.sample(capped, min(N_EVAL, len(capped))), key=lambda r: r["id"])
    preference = sorted(rng.sample(records, min(N_PREFERENCE_PAIRS, len(records))), key=lambda r: r["id"])
    for row in generation + preference:
        row["fair"] = not has_placeholder(row)
    SAMPLE = {"generation": generation, "preference": preference, "leak_check": leak,
              "pool": {"test_exchanges": len(unique), "after_leak_check": len(records), "conversations": len(by_group)}}
    with open(SAMPLE_PATH, "w", encoding="utf-8") as f:
        json.dump(SAMPLE, f, ensure_ascii=False)

GEN, PREF = SAMPLE["generation"], SAMPLE["preference"]
V1_LEAK = SAMPLE["leak_check"]
print(f"Test exchanges: {SAMPLE['pool']['test_exchanges']} -> {SAMPLE['pool']['after_leak_check']} after the v1 leakage check "
      f"({V1_LEAK['removed']} removed{'' if V1_LEAK['checked'] else ', NOT checked'}) from {SAMPLE['pool']['conversations']} conversations")
print(f"Generation sample: {len(GEN)} ({sum(r['fair'] for r in GEN)} without placeholders) | "
      f"preference pairs: {len(PREF)} ({sum(r['fair'] for r in PREF)} without placeholders)")

## C. Each system answers, and is scored

For every system, on the same conversations:
- **Reply** with the deployed decoding and the reply filter. Every system uses the **same random seed per conversation**, so differences come from the models, not from luck in sampling.
- **Likelihood of your real reply:** average negative log-likelihood per token (lower is better), including the end token, so knowing when to stop counts too.
- **Preference:** does the model give your real reply a higher average log-probability than the wrong one?

One base model is in memory at a time. The adapter is put on top, and `disable_adapter()` gives the plain base back exactly. A check proves each adapter really changes its base. Results go to disk per conversation, so a disconnect loses nothing.

In [ ]:
def chat_ids(tokenizer, messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)


@torch.no_grad()
def mean_logprob(model, tokenizer, messages, reply, end_id):
    prompt = chat_ids(tokenizer, messages)["input_ids"][0].tolist()
    target = tokenizer(reply, add_special_tokens=False)["input_ids"] + [end_id]
    ids = torch.tensor([prompt + target], device=DEVICE)
    logits = model(input_ids=ids).logits[0, len(prompt) - 1 : -1].float()
    return torch.log_softmax(logits, -1).gather(1, ids[0, len(prompt):, None]).mean().item()


@torch.no_grad()
def generate(model, tokenizer, messages, end_id, pad_id):
    encoded = chat_ids(tokenizer, messages)
    output = model.generate(**encoded, eos_token_id=end_id, pad_token_id=pad_id, **DECODING)
    new = output[0, encoded["input_ids"].shape[1]:].tolist()
    return tokenizer.decode(new, skip_special_tokens=True).strip(), end_id not in new and len(new) >= DECODING["max_new_tokens"]


def outputs_path(name):
    return f"{EVAL_DIR}/outputs_{name}_private.jsonl"


def load_rows(path):
    rows = {}
    if os.path.exists(path):
        for row in load_jsonl(path):
            rows[row["id"]] = row
    return rows


def run_system(name, model, tokenizer, end_id, pad_id):
    done = load_rows(outputs_path(name))
    todo = [(i, r) for i, r in enumerate(GEN) if r["id"] not in done]
    pref_todo = [r for r in PREF if f"pref:{r['id']}" not in done]
    print(f"{name}: {len(GEN) - len(todo)} replies already done, {len(todo)} to go; {len(pref_todo)} preference pairs to score")
    with open(outputs_path(name), "a", encoding="utf-8") as f:
        for i, record in todo:
            torch.manual_seed(SEED + i)   # the same seed for every system
            raw, hit_limit = generate(model, tokenizer, record["messages"], end_id, pad_id)
            first = iter([raw])
            shown, info = safe_reply(lambda: next(first, None) or generate(model, tokenizer, record["messages"], end_id, pad_id)[0],
                                     find_names, tries=FILTER_TRIES, rng=random.Random(SEED + i))
            nll = -mean_logprob(model, tokenizer, record["messages"], record["real"], end_id)
            f.write(json.dumps({"id": record["id"], "raw": raw, "shown": shown, "hit_limit": hit_limit,
                                "fallback": info["fallback"], "rejected": info["rejected"], "real_nll": nll},
                               ensure_ascii=False) + "\n")
            f.flush()
        for record in pref_todo:
            chosen = mean_logprob(model, tokenizer, record["messages"], record["real"], end_id)
            rejected = mean_logprob(model, tokenizer, record["messages"], record["negative"], end_id)
            f.write(json.dumps({"id": f"pref:{record['id']}", "win": chosen > rejected}) + "\n")
            f.flush()


ADAPTER_EFFECTS = {}
for base in dict.fromkeys(b for b, _ in ACTIVE.values()):
    names = [n for n, (b, _) in ACTIVE.items() if b == base]
    if all(len(load_rows(outputs_path(n))) == len(GEN) + len(PREF) for n in names):
        print(f"{', '.join(names)}: complete")
        continue
    adapters = {n: a for n, (b, a) in ACTIVE.items() if b == base and a}
    assert len(adapters) <= 1, f"One adapter per base model is supported, found {list(adapters)}"
    tokenizer = AutoTokenizer.from_pretrained(base)
    end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    assert end_id is not None and end_id != tokenizer.unk_token_id, f"{base}: the tokenizer has no <|im_end|> token."
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else end_id
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
    if adapters:
        (adapter_name, adapter_path), = adapters.items()
        probe = chat_ids(tokenizer, GEN[0]["messages"])
        with torch.no_grad():
            before = model(**probe).logits[0, -1].float()
        model = PeftModel.from_pretrained(model, adapter_path)
        with torch.no_grad():
            after = model(**probe).logits[0, -1].float()
        ADAPTER_EFFECTS[adapter_name] = (after - before).abs().max().item()
        assert ADAPTER_EFFECTS[adapter_name] > 1e-4, f"{adapter_name}: the adapter changes nothing: it did not load."
        print(f"{adapter_name}: adapter active (largest logit change {ADAPTER_EFFECTS[adapter_name]:.3f})")
    model.eval()
    for name in names:
        plain = isinstance(model, PeftModel) and ACTIVE[name][1] is None
        with model.disable_adapter() if plain else contextlib.nullcontext():
            run_system(name, model, tokenizer, end_id, pad_id)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

OUT = {name: load_rows(outputs_path(name)) for name in ACTIVE}
for name, rows in OUT.items():
    assert all(r["id"] in rows for r in GEN) and all(f"pref:{r['id']}" in rows for r in PREF), f"{name} is incomplete: run again."
print("All systems complete.")

## D. Objective and automatic results

**Statistics.** Every comparison is paired (same conversations). Two systems are compared only on the conversations where they differ: a **sign test** for likelihood, **McNemar's test** for preference (the same idea: only flipped pairs count). With several tests, one can look significant by luck, so the p-values that answer the same question are **Holm-corrected**.

In [ ]:
def sign_test(wins_a, wins_b):
    n = wins_a + wins_b
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins_a, wins_b) + 1)) / 2 ** n) if n else 1.0


def wilson(k, n, z=1.96):
    if n == 0:
        return [0.0, 1.0]
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return [round(max(0.0, centre - half), 3), round(min(1.0, centre + half), 3)]


def holm(p_values):
    order = sorted(p_values, key=p_values.get)
    adjusted, running = {}, 0.0
    for rank, key in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * p_values[key]))
        adjusted[key] = running
    return adjusted


def ids_for(subset, rows):
    return [r["id"] for r in rows if subset == "all" or r["fair"]]


def compare_likelihood(a, b, subset):
    ids = ids_for(subset, GEN)
    a_better = sum(OUT[a][i]["real_nll"] < OUT[b][i]["real_nll"] for i in ids)
    b_better = sum(OUT[a][i]["real_nll"] > OUT[b][i]["real_nll"] for i in ids)
    return {"n": len(ids), "a_mean_nll": round(statistics.mean(OUT[a][i]["real_nll"] for i in ids), 4),
            "b_mean_nll": round(statistics.mean(OUT[b][i]["real_nll"] for i in ids), 4),
            "a_better": a_better, "b_better": b_better, "p": sign_test(a_better, b_better)}


def compare_preference(a, b, subset):
    ids = [f"pref:{i}" for i in ids_for(subset, PREF)]
    wins_a = [OUT[a][i]["win"] for i in ids]
    wins_b = [OUT[b][i]["win"] for i in ids]
    a_only = sum(x and not y for x, y in zip(wins_a, wins_b))
    b_only = sum(y and not x for x, y in zip(wins_a, wins_b))
    return {"n": len(ids), "a_accuracy": round(sum(wins_a) / len(ids), 4), "b_accuracy": round(sum(wins_b) / len(ids), 4),
            "a_only": a_only, "b_only": b_only, "p": sign_test(a_only, b_only)}


PAIRINGS = [("dpo_v2", "sft_v2", "all", "Q1"), ("dpo_v2", "dpo_v1", "fair", "Q2"),
            ("sft_v2", "sft_v1", "fair", "Q2"), ("dpo_v1", "sft_v1", "fair", "context")]
PAIRINGS = [p for p in PAIRINGS if p[0] in ACTIVE and p[1] in ACTIVE]
OBJECTIVE = {}
for a, b, subset, question in PAIRINGS:
    OBJECTIVE[f"{a} vs {b}"] = {"question": question, "subset": subset,
                                "likelihood": compare_likelihood(a, b, subset), "preference": compare_preference(a, b, subset)}

print("Likelihood of your real reply (lower NLL is better) and preference accuracy (higher is better):")
for key, row in OBJECTIVE.items():
    lik, pref = row["likelihood"], row["preference"]
    print(f"\n{key}  [{row['subset']}]")
    print(f"  NLL {lik['a_mean_nll']} vs {lik['b_mean_nll']}: first better in {lik['a_better']}, second in {lik['b_better']} "
          f"(n={lik['n']}, p={lik['p']:.4f})")
    print(f"  preference {pref['a_accuracy']:.1%} vs {pref['b_accuracy']:.1%}: fixed {pref['a_only']}, broken {pref['b_only']} "
          f"(n={pref['n']}, p={pref['p']:.4f})")

In [ ]:
GENERIC_AI_PHRASES = ["ذكاء اصطناعي", "كمساعد", "أنا هنا عشان أساعدك", "لو عندك أي سؤال", "كيف يمكنني مساعدتك", "متترددش تسألني"]
DARIJA_MARKERS = ["شنو", "بغيتي", "ديالي", "واخا", "بزاف"]


def char_f1(candidate, reference, n=3):
    def grams(t):
        t = re.sub(r"\s+", "", t)
        return Counter(t[i:i + n] for i in range(len(t) - n + 1)) if len(t) >= n else Counter([t] if t else [])
    c, r = grams(candidate), grams(reference)
    overlap = sum((c & r).values())
    if not overlap:
        return 0.0
    precision, recall = overlap / sum(c.values()), overlap / sum(r.values())
    return 2 * precision * recall / (precision + recall)


def normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def describe(texts):
    return {"median_chars": statistics.median(len(t) for t in texts),
            "with_placeholder": sum(bool(PLACEHOLDER.search(t)) for t in texts),
            "with_private_name": sum(bool(find_names and find_names(t)) for t in texts),
            "with_long_number": sum(bool(LONG_NUMBER.search(t)) for t in texts),
            "generic_ai_phrasing": sum(any(p in t for p in GENERIC_AI_PHRASES) for t in texts),
            "darija": sum(any(p in t for p in DARIJA_MARKERS) for t in texts)}


AUTOMATIC = {"real": {"all": describe([r["real"] for r in GEN])}}
for name in ACTIVE:
    rows = [OUT[name][r["id"]] for r in GEN]
    AUTOMATIC[name] = {
        "raw": describe([row["raw"] for row in rows]),
        "shown": describe([row["shown"] for row in rows]),
        "hit_token_limit": sum(row["hit_limit"] for row in rows),
        "rejected_first_try": sum(bool(row["rejected"]) for row in rows),
        "fallbacks": sum(row["fallback"] for row in rows),
        "char_overlap_with_real": round(statistics.mean(char_f1(OUT[name][r["id"]]["shown"], r["real"]) for r in GEN), 4),
        "identical_to_dpo_v2": sum(normalize(OUT[name][r["id"]]["shown"]) == normalize(OUT["dpo_v2"][r["id"]]["shown"]) for r in GEN),
    }

print(f"Real replies: median {AUTOMATIC['real']['all']['median_chars']} characters")
header = ["median chars", "placeholders (raw)", "names (raw)", "names (shown)", "hit limit", "fallbacks", "char overlap", "same as DPO v2"]
print(f"{'':8s}" + " | ".join(header))
for name in ACTIVE:
    row = AUTOMATIC[name]
    values = [row["shown"]["median_chars"], row["raw"]["with_placeholder"], row["raw"]["with_private_name"],
              row["shown"]["with_private_name"], row["hit_token_limit"], row["fallbacks"], row["char_overlap_with_real"],
              row["identical_to_dpo_v2"]]
    print(f"{name:8s}" + " | ".join(f"{v!s:>{len(h)}}" for v, h in zip(values, header)))

## E. The local judge

**Two questions:**
- **sense:** which reply is the better next message here (makes sense, answers what was said, sounds like a person chatting, not an assistant)? The judge does not see your real reply.
- **like:** which reply is closer to the reply you really sent?

**How it answers.** Instead of writing text, the judge's **probabilities** for "A" and "B" as its first answer token are read: one forward pass, nothing to parse, and a confidence for free. Every pair is judged **in both orders**. A system wins a conversation only if both orders agree; otherwise it's a tie. This cancels the judge's habit of preferring the first (or second) option.

**Calibration first.** Before its verdicts count, the judge must pick **your real reply over the wrong one** (the DPO hard negative) in at least `MIN_JUDGE_CALIBRATION` of the conversations. If it can't, it can't tell good from bad here, and its verdicts are reported but **not used** for the conclusions.

**Comparisons**, only those that answer the questions: calibration; DPO v2 vs SFT v2 (sense, like); DPO v2 vs DPO v1 (sense, like; fair subset); your real reply vs DPO v2 and vs SFT v2 (sense: how close to a person).

In [ ]:
SENSE_PROMPT = (
    "You are reading a WhatsApp chat in Egyptian Arabic between Ahmed and another person. Below is the recent "
    "conversation and two possible next messages from Ahmed.\n"
    "Which one is the better next message: it makes sense here, responds to what was just said, and reads like a "
    "real person chatting (not like an assistant)? Do not prefer a reply for being longer or more polite.\n\n"
    "CONVERSATION:\n{conversation}\n\nREPLY A: {a}\nREPLY B: {b}\n\nAnswer with one letter, A or B."
)
LIKE_PROMPT = (
    "You are reading a WhatsApp chat in Egyptian Arabic between Ahmed and another person. Below is the recent "
    "conversation, the REAL reply Ahmed actually sent, and two candidate replies.\n"
    "Which candidate is closer to Ahmed's real reply in meaning, tone and length? Do not prefer a candidate for being "
    "longer, more polite or more correct.\n\n"
    "CONVERSATION:\n{conversation}\n\nREAL REPLY: {real}\n\nCANDIDATE A: {a}\nCANDIDATE B: {b}\n\n"
    "Answer with one letter, A or B."
)


def conversation_text(record):
    turns = [m for m in record["messages"] if m["role"] != "system"][-JUDGE_CONTEXT_TURNS:]
    return "\n".join(f"{'Ahmed' if m['role'] == 'assistant' else 'Other person'}: {m['content']}" for m in turns)


def reply_of(side, record):
    # Every text as a reader would see it: model replies are already filtered, so the real and wrong replies
    # lose their placeholders too; otherwise "[NAME]" alone would tell the judge which reply is real.
    if side == "real":
        return clean_reply(record["real"]) or record["real"]
    if side == "negative":
        return clean_reply(record["negative"]) or record["negative"]
    return OUT[side][record["id"]]["shown"]


COMPARISONS = [
    ("calibration", "sense", "real", "negative", "all"),
    ("Q1 sense", "sense", "dpo_v2", "sft_v2", "all"),
    ("Q1 like", "like", "dpo_v2", "sft_v2", "all"),
    ("Q2 sense", "sense", "dpo_v2", "dpo_v1", "fair"),
    ("Q2 like", "like", "dpo_v2", "dpo_v1", "fair"),
    ("Q4 real vs dpo_v2", "sense", "real", "dpo_v2", "all"),
    ("Q4 real vs sft_v2", "sense", "real", "sft_v2", "all"),
]
COMPARISONS = [c for c in COMPARISONS if all(s in ACTIVE or s in ("real", "negative") for s in (c[2], c[3]))]
JUDGE_PATH = f"{EVAL_DIR}/judge_{JUDGE_TAG}_private.jsonl"
print("Judge comparisons:", ", ".join(c[0] for c in COMPARISONS))

In [ ]:
if RUN_JUDGE:
    for name in ("model", "tokenizer"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    judge_tokenizer.padding_side = "left"
    if judge_tokenizer.pad_token_id is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if JUDGE_LOAD_4BIT and DEVICE == "cuda" else None
    judge = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL, dtype=torch.float16 if DEVICE == "cuda" else torch.float32, quantization_config=quantization,
        device_map={"": 0} if DEVICE == "cuda" else None)
    judge.eval()

    def option_ids(letter):
        ids = set()
        for variant in (letter, " " + letter):
            encoded = judge_tokenizer(variant, add_special_tokens=False)["input_ids"]
            if len(encoded) == 1:
                ids.add(encoded[0])
        assert ids, f"'{letter}' is not a single token for this judge"
        return sorted(ids)

    A_IDS, B_IDS = option_ids("A"), option_ids("B")

    def judge_text(prompt):
        messages = [{"role": "user", "content": prompt}]
        return judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    @torch.no_grad()
    def prob_first_option(prompts):
        # P(answer is A), from the judge's next-token probabilities, renormalized over A and B.
        encoded = judge_tokenizer([judge_text(p) for p in prompts], return_tensors="pt", padding=True,
                                  add_special_tokens=False).to(judge.device)
        positions = (encoded["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        logits = judge(**encoded, position_ids=positions, logits_to_keep=1).logits[:, -1].float()
        return torch.sigmoid(torch.logsumexp(logits[:, A_IDS], -1) - torch.logsumexp(logits[:, B_IDS], -1)).tolist()

    def build_prompt(question, record, first, second):
        template = SENSE_PROMPT if question == "sense" else LIKE_PROMPT
        return template.format(conversation=conversation_text(record), real=reply_of("real", record), a=first, b=second)

    done = {(row["comparison"], row["id"]) for row in load_jsonl(JUDGE_PATH)} if os.path.exists(JUDGE_PATH) else set()
    jobs = []
    for label, question, a, b, subset in COMPARISONS:
        for record in GEN:
            if (subset == "fair" and not record["fair"]) or (label, record["id"]) in done:
                continue
            text_a, text_b = reply_of(a, record), reply_of(b, record)
            if normalize(text_a) == normalize(text_b):
                jobs.append((label, record["id"], None, None, None))
            else:
                jobs.append((label, record["id"], build_prompt(question, record, text_a, text_b),
                             build_prompt(question, record, text_b, text_a), None))
    print(f"Judge: {len(done)} verdicts saved already, {len(jobs)} to go")
    with open(JUDGE_PATH, "a", encoding="utf-8") as f:
        for start in range(0, len(jobs), JUDGE_BATCH):
            batch = jobs[start:start + JUDGE_BATCH]
            prompts = [p for job in batch if job[2] for p in (job[2], job[3])]
            probs = iter(prob_first_option(prompts)) if prompts else iter([])
            for label, record_id, first, second, _ in batch:
                if first is None:
                    row = {"comparison": label, "id": record_id, "winner": "tie", "identical": True}
                else:
                    p_a_first, p_b_first = next(probs), next(probs)    # order 1: a shown as A; order 2: b shown as A
                    a_votes = (p_a_first > 0.5) + (p_b_first < 0.5)
                    winner = "a" if a_votes == 2 else ("b" if a_votes == 0 else "tie")
                    row = {"comparison": label, "id": record_id, "winner": winner, "identical": False,
                           "p_a_first": round(p_a_first, 4), "p_b_first": round(p_b_first, 4),
                           "a_score": round((p_a_first + 1 - p_b_first) / 2, 4)}
                f.write(json.dumps(row) + "\n")
            f.flush()
    del judge
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("RUN_JUDGE = False: the conclusions use the objective evidence only.")

In [ ]:
JUDGE = {}
if RUN_JUDGE:
    verdicts = {}
    for row in load_jsonl(JUDGE_PATH):
        verdicts[(row["comparison"], row["id"])] = row
    for label, question, a, b, subset in COMPARISONS:
        rows = [verdicts[(label, r["id"])] for r in GEN if (label, r["id"]) in verdicts]
        judged = [r for r in rows if not r["identical"]]
        a_wins, b_wins = sum(r["winner"] == "a" for r in rows), sum(r["winner"] == "b" for r in rows)
        JUDGE[label] = {
            "question": question, "a": a, "b": b, "subset": subset, "n": len(rows), "identical": len(rows) - len(judged),
            "a_wins": a_wins, "b_wins": b_wins, "ties": len(rows) - a_wins - b_wins, "p": sign_test(a_wins, b_wins),
            "a_share_of_decided_ci95": wilson(a_wins, a_wins + b_wins),
            "a_mean_score": round(statistics.mean(r["a_score"] for r in judged), 4) if judged else None,
            "order_consistency": round(sum(r["winner"] != "tie" for r in judged) / len(judged), 3) if judged else None,
            "first_position_bias": round(statistics.mean((r["p_a_first"] + r["p_b_first"]) / 2 for r in judged), 3) if judged else None,
        }
    calibration = JUDGE.get("calibration", {})
    CALIBRATION_ACCURACY = calibration["a_wins"] / calibration["n"] if calibration.get("n") else 0.0
    JUDGE_TRUSTED = CALIBRATION_ACCURACY >= MIN_JUDGE_CALIBRATION
    print(f"Calibration: the judge picked your real reply over a wrong one in {CALIBRATION_ACCURACY:.0%} "
          f"(needs {MIN_JUDGE_CALIBRATION:.0%}) -> judge {'TRUSTED' if JUDGE_TRUSTED else 'NOT trusted: its verdicts are not used'}")
    for label, row in JUDGE.items():
        print(f"  {label:20s} {row['a']} {row['a_wins']} | {row['b']} {row['b_wins']} | ties {row['ties']} "
              f"(identical {row['identical']}) | p={row['p']:.4f} | consistency {row['order_consistency']} | "
              f"first-position bias {row['first_position_bias']}")
else:
    CALIBRATION_ACCURACY, JUDGE_TRUSTED = None, False

## F. Answers to the four questions

Each line of evidence is classified as **first better**, **second better** or **no significant difference** (Holm-corrected p ≥ 0.05). The judge's lines count only if it passed calibration.

In [ ]:
def classify(a, b, a_count, b_count, p):
    if p >= 0.05:
        return "no significant difference"
    return f"{a} better" if a_count > b_count else f"{b} better"


EVIDENCE = {}
for key, row in OBJECTIVE.items():
    a, b = key.split(" vs ")
    EVIDENCE[f"{row['question']} | likelihood | {key}"] = (a, b, row["likelihood"]["a_better"], row["likelihood"]["b_better"], row["likelihood"]["p"])
    EVIDENCE[f"{row['question']} | preference | {key}"] = (a, b, row["preference"]["a_only"], row["preference"]["b_only"], row["preference"]["p"])
if JUDGE_TRUSTED:
    for label, row in JUDGE.items():
        if label.startswith(("Q1", "Q2")):
            EVIDENCE[f"{label[:2]} | judge {row['question']} | {row['a']} vs {row['b']}"] = (row["a"], row["b"], row["a_wins"], row["b_wins"], row["p"])

ADJUSTED = {}
for question in ("Q1", "Q2", "context"):
    keys = [k for k in EVIDENCE if k.startswith(question)]
    ADJUSTED.update(holm({k: EVIDENCE[k][4] for k in keys}))
FINDINGS = {k: classify(*EVIDENCE[k][:4], ADJUSTED[k]) for k in EVIDENCE}
for key, finding in FINDINGS.items():
    print(f"  {key:45s} {finding:28s} (Holm p={ADJUSTED[key]:.4f})")


def answer(question, first, second):
    found = [v for k, v in FINDINGS.items() if k.startswith(question) and f"{first} vs {second}" in k]
    if not found:
        return "not measured (a system is missing)"
    better, worse = found.count(f"{first} better"), found.count(f"{second} better")
    if better and not worse:
        return f"YES: {first} is better on {better} of {len(found)} measures, worse on none"
    if worse and not better:
        return f"NO: {second} is better on {worse} of {len(found)} measures"
    if better and worse:
        return f"MIXED: better on {better}, worse on {worse} of {len(found)} measures; read the examples"
    return "NO SIGNIFICANT DIFFERENCE on this sample"


shown = AUTOMATIC["dpo_v2"]["shown"]
ANSWERS = {
    "Q1 Did DPO v2 improve on SFT v2?": answer("Q1", "dpo_v2", "sft_v2"),
    "Q2 Is v2 better than the old final model (DPO v1)?": answer("Q2", "dpo_v2", "dpo_v1"),
    "Q3 Is DPO v2 safe to show?": ("YES" if shown["with_private_name"] == 0 and shown["with_placeholder"] == 0
                                   and AUTOMATIC["dpo_v2"]["fallbacks"] < 0.1 * len(GEN) else "NO") +
                                  f": shown replies with a name {shown['with_private_name']}, with a placeholder "
                                  f"{shown['with_placeholder']}, fallbacks {AUTOMATIC['dpo_v2']['fallbacks']} of {len(GEN)}",
}
if RUN_JUDGE and "Q4 real vs dpo_v2" in JUDGE:
    row = JUDGE["Q4 real vs dpo_v2"]
    ANSWERS["Q4 How close to a real person?"] = (
        f"the judge preferred your real reply in {row['a_wins']}, DPO v2's in {row['b_wins']}, tie {row['ties']} of {row['n']}"
        + ("" if JUDGE_TRUSTED else " (judge NOT trusted)"))
for question, text in ANSWERS.items():
    print(f"\n{question}\n  {text}")

## G. Optional: your own blind check of the judge

A sample of conversations where DPO v2 and SFT v2 replied differently, as A and B in a random order. Vote which reply makes more sense, like in the 30-message test. The cell then reports **how often the judge agrees with you**. A judge that agrees with you most of the time can be trusted on the 200; one that doesn't is just noise. This is the only check of the judge against a human.

In [ ]:
different = [r for r in GEN if normalize(OUT["dpo_v2"][r["id"]]["shown"]) != normalize(OUT["sft_v2"][r["id"]]["shown"])]
HUMAN_SAMPLE = random.Random(SEED + 1).sample(different, min(HUMAN_CHECK_SIZE, len(different)))
DPO_IS_A = [random.Random(SEED + 2 + i).random() < 0.5 for i in range(len(HUMAN_SAMPLE))]
for i, record in enumerate(HUMAN_SAMPLE):
    first, second = ("dpo_v2", "sft_v2") if DPO_IS_A[i] else ("sft_v2", "dpo_v2")
    print(f"{i + 1:2d}. " + conversation_text(record).replace("\n", "\n    "))
    print(f"   A: {OUT[first][record['id']]['shown']}\n   B: {OUT[second][record['id']]['shown']}\n")

In [ ]:
VOTES = ""   # one character per conversation above: A, B or =

votes = [v for v in VOTES.upper() if v in "AB="]
HUMAN = None
if not votes:
    print(f"Optional: fill VOTES with {len(HUMAN_SAMPLE)} characters (A, B or =) and run this cell.")
else:
    assert len(votes) == len(HUMAN_SAMPLE), f"{len(votes)} votes for {len(HUMAN_SAMPLE)} conversations."
    human = {r["id"]: ("tie" if v == "=" else ("dpo_v2" if (v == "A") == DPO_IS_A[i] else "sft_v2"))
             for i, (r, v) in enumerate(zip(HUMAN_SAMPLE, votes))}
    agree = total = 0
    if RUN_JUDGE:
        for record_id, choice in human.items():
            row = verdicts.get(("Q1 sense", record_id))
            if row and choice != "tie" and row["winner"] != "tie":
                total += 1
                agree += (row["winner"] == "a") == (choice == "dpo_v2")
    HUMAN = {"dpo_v2": sum(v == "dpo_v2" for v in human.values()), "sft_v2": sum(v == "sft_v2" for v in human.values()),
             "ties": sum(v == "tie" for v in human.values()), "judge_agreement": [agree, total]}
    print(f"You preferred DPO v2 in {HUMAN['dpo_v2']}, SFT v2 in {HUMAN['sft_v2']}, tie {HUMAN['ties']}")
    print(f"The judge agreed with you on {agree} of {total} conversations where you both picked a side"
          + (f" ({agree / total:.0%})" if total else ""))

## H. Save, and examples

`eval_summary.json` holds **numbers only**: settings, counts, p-values, answers. It is safe to share; the next cell checks that no reply text got into it. Everything with text stays in the `_private` files.

In [ ]:
SUMMARY = json.loads(json.dumps({
    "eval_run": EVAL_RUN_NAME, "config": GEN_CONFIG, "judge_model": JUDGE_MODEL if RUN_JUDGE else None,
    "sample": {"generation": len(GEN), "generation_fair": sum(r["fair"] for r in GEN), "preference": len(PREF),
               "preference_fair": sum(r["fair"] for r in PREF), "v1_leak_check": V1_LEAK, "pool": SAMPLE["pool"]},
    "adapter_effects": ADAPTER_EFFECTS, "objective": OBJECTIVE, "automatic": AUTOMATIC,
    "judge": {"calibration_accuracy": CALIBRATION_ACCURACY, "trusted": JUDGE_TRUSTED, "comparisons": JUDGE},
    "findings": {k: {"finding": v, "holm_p": ADJUSTED[k]} for k, v in FINDINGS.items()},
    "answers": ANSWERS, "human_check": HUMAN,
}))
summary_text = json.dumps(SUMMARY, ensure_ascii=False)
leaked = [r["id"] for r in GEN if r["real"].strip() and len(r["real"]) > 3 and r["real"] in summary_text]
assert not leaked, "Reply text found in the shareable summary."
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(SUMMARY, f, indent=2, ensure_ascii=False)
with open(PRIVATE_PATH, "w", encoding="utf-8") as f:
    for record in GEN:
        f.write(json.dumps({"id": record["id"], "fair": record["fair"], "context": record["messages"][1:], "real": record["real"],
                            **{name: OUT[name][record["id"]]["shown"] for name in ACTIVE}}, ensure_ascii=False) + "\n")
print("Saved:", SUMMARY_PATH, "(numbers only) |", PRIVATE_PATH, "(private)")

if SHOW_EXAMPLES:
    for record in GEN[:10]:
        print("\n" + conversation_text(record) + f"\n  REAL: {record['real']}")
        for name in ACTIVE:
            print(f"  {name}: {OUT[name][record['id']]['shown']}")

## Known limitations
- **One judge model.** Qwen3-14B reads Egyptian Arabic well but not perfectly. The calibration test and your blind check (G) show how far to trust it.
- **Few conversations.** About 77 test conversations, at most 4 exchanges each. Exchanges from one conversation are not fully independent, so the p-values are a little optimistic.
- **v1 comparisons are on the fair subset** (no placeholders) and depend on the leakage check. If `train.jsonl` was missing, v1 may look better than it is.
- **Sampling.** The deployed decoding samples, so a single run has some luck in it; the shared seed makes it fair between systems, not noise-free.
- **The judge's "sense" is not "true".** It can't know facts about your life, so it judges plausibility, not correctness.